# Provenance-Locked EEGPT Frozen Probe

**STOPPED COMPLETED MAINTAINED-CONVERSION CONFIGURATION.** The pinned Braindecode encoder-only probe reached 50.85% Full50 BA. It is not the blocked original BINE022 protocol. Do not rerun or tune this exact mapping; workspace-root `AGENTS.md` section 2e is authoritative. Execution fails closed unless `allow_closed_rerun` is explicitly overridden after a new prespecification.

# 1. Setup

In [ ]:
import hashlib, json, platform, sys
from datetime import datetime
from pathlib import Path
WORKING_DIR = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'src' / 'liu2024').is_dir())
sys.path.insert(0, str(WORKING_DIR / 'src' / 'liu2024'))
from liu2024_candidate_runner import run_candidate_experiment
print(f'Python: {sys.version.split()[0]} | Platform: {platform.platform()} | Root: {WORKING_DIR}')

# 2. Configuration
## 2.1 CONFIG
This notebook intentionally fails closed until local official assets and their exact digest are supplied.

In [ ]:
CONFIG = {
    'allow_closed_rerun': False,
    # Paths / run identity
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-eegpt-frozen-probe'),
    'resume_run_dir': None,
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'split_manifest_path': str(WORKING_DIR / 'artifacts' / 'liu2024-compact-mi-models' / '20260712_165746_790145_fd8ab986' / 'splits.json'),
    'experiment_name': 'eegpt_frozen_probe',
    'candidate_family': 'frozen_eegpt_probe',
    'config_note': 'Official EEGPT target encoder, mean-pooled patch summaries, fold-local shrinkage LDA.',
    # Pinned maintained Braindecode conversion of the official encoder
    'eegpt_backend': 'braindecode_encoder_only',
    'hub_repo_id': 'braindecode/eegpt-pretrained',
    'hub_revision': 'e41cb3ae2ce4fd9eb736862292c91f8128d15618',
    'checkpoint_dir': str(WORKING_DIR / 'artifacts' / 'checkpoint_cache' / 'eegpt_braindecode_e41cb3ae'),
    'checkpoint_config_sha256': '9af22c210ca9ac00970f749c00153465ab8cecefce0336111986ec9d13af9260',
    'checkpoint_sha256': 'fb34c20609983324679b9534f9d17a2289a232d0276dd2b8b7d5b088f876f621',
    # Dataset / preprocessing
    'subjects_to_use': None,
    'target_sfreq': 250,
    'mi_window_seconds': 4.0,
    'average_reference': True,
    'bandpass_hz': [0.5, 40.0],
    'feature_batch_size': 8,
    # Evaluation / reproducibility
    'seed': 2026,
}


## 2.2 Artifact Creation and Reproducibility

In [ ]:
if not CONFIG.get('allow_closed_rerun', False):
    raise RuntimeError('CLOSED: maintained EEGPT Full50 is complete and negative; see AGENTS.md section 2e.')
missing = [key for key in ['checkpoint_dir', 'checkpoint_config_sha256', 'checkpoint_sha256'] if not CONFIG[key]]
if missing:
    raise RuntimeError(f'EEGPT assets are not configured; refusing to run with guessed or partial provenance: {missing}')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:8]
ARTIFACT_DIR = Path(CONFIG['resume_run_dir']).resolve() if CONFIG.get('resume_run_dir') else Path(CONFIG['artifact_dir']) / RUN_ID
if CONFIG.get('resume_run_dir'):
    if not ARTIFACT_DIR.is_dir(): raise FileNotFoundError(f'Resume directory does not exist: {ARTIFACT_DIR}')
    RUN_ID = ARTIFACT_DIR.name
else:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
(ARTIFACT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')

# 3. Load and Prepare Data
Exact marker-2 crops are independently filtered and resampled to 29 x 1024. Legacy temporal names map T3/T4/T5/T6 to T7/T8/P7/P8.
# 4. Model
Only official `target_encoder.*` keys are loaded; missing, unexpected, or shape-mismatched keys abort the run.
# 5. Training
The encoder is frozen. Scaling and shrinkage LDA fit on each outer-training fold only.
# 6. Results

In [ ]:
try:
    RUN_METADATA = run_candidate_experiment(CONFIG, ARTIFACT_DIR)
    (ARTIFACT_DIR / 'run.log').write_text('completed\n', encoding='utf-8')
except Exception as error:
    (ARTIFACT_DIR / 'run.log').write_text(f'failed: {type(error).__name__}: {error}\n', encoding='utf-8')
    raise
print(json.dumps(RUN_METADATA['global_metrics'], indent=2))
print(f'\nAll artifacts in: {ARTIFACT_DIR}')